In [3]:
import os
import base64
import ollama
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# ==========================================
# 📄 PHASE 1: VISION PROMPT (The Eyes)
# ==========================================
CLINICAL_EXTRACTION_PROMPT = """
Analyze this medical X-ray with extreme precision for a downstream Reasoning LLM. 
Extract and list the following attributes in a structured, technical format:
1. BONE(S) INVOLVED: Identify specific bones.
2. FRACTURE PRESENCE: [Yes/No/Inconclusive].
3. MORPHOLOGY: (e.g., Transverse, Oblique, Spiral, Comminuted, Greenstick).
4. LOCATION: Specific segment (e.g., Intra-articular, Mid-shaft, Proximal).
5. DISPLACEMENT: Mention percentage and direction.
6. ANGULATION: Degree and direction if visible.
7. SOFT TISSUE: Note any significant swelling or joint effusion.
8. CONFIDENCE SCORE: 0-100% based on image clarity.
Provide ONLY the technical extraction. Do not provide patient advice.
"""

# ==========================================
# 📄 PHASE 2: REASONING PROMPT (The Brain)
# ==========================================
REASONING_SYSTEM_PROMPT = """
ACT AS: A Consultant Orthopedic Surgeon.
CONTEXT: You are reviewing a technical extraction report provided by a Radiology AI. 
YOU DO NOT HAVE ACCESS TO THE ORIGINAL IMAGE. Your task is to interpret the text-based 
clinical features to finalize a management plan.

REQUIRED REPORT SECTIONS:
1. CLINICAL SYNTHESIS: Summarize the findings into a concise diagnosis.
2. TRIAGE CATEGORY: Classify as [EMERGENT], [URGENT], or [ROUTINE] based on the data.
3. PATHOPHYSIOLOGY: Explain the implications of the morphology (e.g., why a 'Spiral' fracture suggests rotation).
4. STABILITY ASSESSMENT: Determine if the features indicate a mechanically unstable fracture.
5. SURGICAL VS. NON-SURGICAL: Provide a recommendation based on orthopedic standards.
"""

# ==========================================
# 📸 PHASE 1 FUNCTIONS: FEATURE EXTRACTION
# ==========================================
def get_groq_vision(image_path):
    client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode('utf-8')
    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": [
            {"type": "text", "text": CLINICAL_EXTRACTION_PROMPT},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
        ]}],
        temperature=0.0
    )
    return response.choices[0].message.content

def get_local_vision(image_path):
    # Optimized for GTX 1650: Clear VRAM immediately
    response = ollama.chat(
        model='qwen3-vl:2b',
        keep_alive=0, 
        messages=[{'role': 'user', 'content': CLINICAL_EXTRACTION_PROMPT, 'images': [image_path]}]
    )
    return response['message']['content']

# ==========================================
# 🧠 PHASE 2 FUNCTIONS: CLINICAL REASONING
# ==========================================
def get_groq_reasoning(features):
    client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    # Using Llama-3-70b for high-speed cloud reasoning
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": REASONING_SYSTEM_PROMPT},
            {"role": "user", "content": f"INPUT FEATURES:\n{features}"}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

def get_local_reasoning(features):
    # Using gemma4:e4b as your local reasoning model
    response = ollama.chat(
        model='gemma4:e4b',
        keep_alive=0,
        messages=[
            {"role": "system", "content": REASONING_SYSTEM_PROMPT},
            {"role": "user", "content": f"INPUT FEATURES:\n{features}"}
        ]
    )
    return response['message']['content']

# ==========================================
# 🚀 THE MASTER PIPELINE (The "Main Chain")
# ==========================================
def nexmed_pipeline(image_path, vision_choice="Groq", reasoning_choice="Groq"):
    # 1. Vision Extraction
    print(f"👀 Phase 1: Extracting features using {vision_choice}...")
    if vision_choice == "Groq":
        features = get_groq_vision(image_path)
    else:
        features = get_local_vision(image_path)
    
    # 2. Reasoning Report
    print(f"🧠 Phase 2: Generating report using {reasoning_choice}...")
    if reasoning_choice == "Groq":
        report = get_groq_reasoning(features)
    else:
        report = get_local_reasoning(features)
        
    return features, report

if __name__ == "__main__":
    img = r"C:\Users\ishaa\Downloads\images (1).jpg"
    features, report = nexmed_pipeline(img, vision_choice="Groq", reasoning_choice="Groq")
    
    print("\n" + "="*50)
    print("📋 PHASE 1 FEATURES:\n", features)
    print("\n" + "="*50)
    print("👨‍⚕️ PHASE 2 REPORT:\n", report)

👀 Phase 1: Extracting features using Groq...
🧠 Phase 2: Generating report using Groq...

📋 PHASE 1 FEATURES:
 1. **BONE(S) INVOLVED**: 
   - Radius
   - Ulna

2. **FRACTURE PRESENCE**: 
   - Yes

3. **MORPHOLOGY**: 
   - Radius: Oblique fracture
   - Ulna: Oblique fracture

4. **LOCATION**: 
   - Radius: Mid-shaft
   - Ulna: Mid-shaft

5. **DISPLACEMENT**: 
   - Radius: Approximately 10-15% displacement in a dorsal direction
   - Ulna: Approximately 5-10% displacement in a dorsal direction

6. **ANGULATION**: 
   - Visible angulation in both bones, approximately 5 degrees in a dorsal direction.

7. **SOFT TISSUE**: 
   - No significant swelling or joint effusion noted.

8. **CONFIDENCE SCORE**: 
   - 90%

👨‍⚕️ PHASE 2 REPORT:
 **1. CLINICAL SYNTHESIS**  
- **Diagnosis:** Closed, diaphyseal (mid‑shaft) oblique fractures of the radius and ulna with mild dorsal displacement (≈10‑15 % radius, 5‑10 % ulna) and ~5° dorsal angulation. No soft‑tissue swelling, joint effusion, or neurovascular 